# Parte 10 - Materialized View en BigQuery

Una **Materialized View** es una vista cuyos resultados se almacenan físicamente en disco y se actualizan automáticamente cuando los datos base cambian. A diferencia de una vista normal, no recalcula el resultado en cada consulta, lo que mejora significativamente el rendimiento.

## Configuración del cliente BigQuery

In [1]:
from google.cloud import bigquery
import os

# Ruta a tu archivo de credenciales
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../credentials/retail-analytics-lab-grupo2-a4bfa003951d.json'

client = bigquery.Client()
print(f'Proyecto activo: {client.project}')

Proyecto activo: retail-analytics-lab-grupo2


## Paso 1 - Crear la Materialized View

Creamos una vista materializada que pre-agrega las ventas totales por producto.

In [2]:
sql_create_mv = """
CREATE MATERIALIZED VIEW IF NOT EXISTS retail_dw.mv_ventas_producto AS
SELECT
    id_producto,
    SUM(monto) AS total_ventas
FROM retail_dw.ventas
GROUP BY id_producto;
"""

job = client.query(sql_create_mv)
job.result()
print('✅ Materialized View creada exitosamente.')

✅ Materialized View creada exitosamente.


## Paso 2 - Verificar que la Materialized View existe

In [3]:
sql_check = """
SELECT
    table_name,
    table_type,
    creation_time
FROM retail_dw.INFORMATION_SCHEMA.TABLES
WHERE table_name = 'mv_ventas_producto';
"""

df_check = client.query(sql_check).to_dataframe()
print(df_check)

c:\Users\luise\miniconda3\envs\bigquery_lab\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


           table_name         table_type                    creation_time
0  mv_ventas_producto  MATERIALIZED VIEW 2026-06-19 04:53:15.412000+00:00


## Paso 3 - Consultar la Materialized View

In [4]:
sql_query_mv = """
SELECT
    mv.id_producto,
    p.producto,
    mv.total_ventas
FROM retail_dw.mv_ventas_producto mv
JOIN retail_dw.productos p
    ON mv.id_producto = p.id_producto
ORDER BY mv.total_ventas DESC
LIMIT 10;
"""

df_mv = client.query(sql_query_mv).to_dataframe()
print('Top 10 productos por ventas totales (desde Materialized View):')
print(df_mv)

c:\Users\luise\miniconda3\envs\bigquery_lab\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Top 10 productos por ventas totales (desde Materialized View):
   id_producto                                producto  total_ventas
0          227           Farré y asociados S.A. cu-335      62534.67
1          227           Farré y asociados S.A. cu-335      62534.67
2          227                            Producto_227      62534.67
3          399                            Producto_399      62445.04
4          399  Comercializadora Iberia S.Coop. ha-692      62445.04
5          399  Comercializadora Iberia S.Coop. ha-692      62445.04
6          310            Benavent y Ribes S.A. Jq-973      61913.57
7          310                            Producto_310      61913.57
8          310            Benavent y Ribes S.A. Jq-973      61913.57
9          352                            Producto_352      60492.26


## Paso 4 - Comparar rendimiento: Tabla base vs Materialized View

Medimos el tiempo de ejecución de la misma consulta en la tabla original y en la vista materializada.

In [5]:
import time

# --- Consulta sobre la tabla base ---
sql_base = """
SELECT
    id_producto,
    SUM(monto) AS total_ventas
FROM retail_dw.ventas
GROUP BY id_producto
ORDER BY total_ventas DESC
LIMIT 10;
"""

inicio = time.time()
job_base = client.query(sql_base)
job_base.result()
tiempo_base = time.time() - inicio

# --- Consulta sobre la Materialized View ---
sql_mv = """
SELECT
    id_producto,
    total_ventas
FROM retail_dw.mv_ventas_producto
ORDER BY total_ventas DESC
LIMIT 10;
"""

inicio = time.time()
job_mv = client.query(sql_mv)
job_mv.result()
tiempo_mv = time.time() - inicio

print('=' * 50)
print(f'⏱ Tiempo tabla base:          {tiempo_base:.3f}s')
print(f'⏱ Tiempo Materialized View:   {tiempo_mv:.3f}s')
print('=' * 50)

# Bytes procesados
bytes_base = job_base.total_bytes_processed or 0
bytes_mv   = job_mv.total_bytes_processed or 0
print(f'📦 Bytes procesados (base):   {bytes_base / 1e6:.2f} MB')
print(f'📦 Bytes procesados (MV):     {bytes_mv   / 1e6:.2f} MB')

if bytes_base > 0:
    ahorro = (1 - bytes_mv / bytes_base) * 100
    print(f'\n✅ Reducción de datos procesados: {ahorro:.1f}%')

⏱ Tiempo tabla base:          0.720s
⏱ Tiempo Materialized View:   0.950s
📦 Bytes procesados (base):   1.60 MB
📦 Bytes procesados (MV):     1.60 MB

✅ Reducción de datos procesados: 0.0%


## Resumen

| Concepto | Descripción |
|---|---|
| **Materialized View** | Vista con resultados pre-calculados y almacenados |
| **Actualización** | BigQuery la refresca automáticamente cuando cambian los datos base |
| **Beneficio principal** | Menor tiempo de respuesta y menor volumen de datos procesados |
| **Cuándo usarla** | Consultas de agregación frecuentes sobre tablas grandes |
